<a href="https://colab.research.google.com/github/abhijith1922005-arch/Mini_project/blob/main/superstore_ETL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')
import os
import pandas as pd
import numpy as np

Mounted at /content/drive


In [2]:
project_path = "/content/drive/MyDrive/Superstore_ETL"

os.makedirs(project_path + "/raw_data", exist_ok=True)
os.makedirs(project_path + "/reports", exist_ok=True)
os.makedirs(project_path + "/screenshots", exist_ok=True)
os.makedirs(project_path + "/output", exist_ok=True)

print("Project folders created successfully")

Project folders created successfully


In [3]:
# EXTRACT

def extract():

    print("Starting data extraction...")

    url = "https://raw.githubusercontent.com/abhijith1922005-arch/Mini_project/3e2ee24b562eeca35ff45e4a5c574645180f5ca0/Sample%20-%20Superstore.csv"

    df = pd.read_csv(url, encoding="latin1")

    print("Data extracted successfully!")
    print("Number of records:", df.shape[0])
    print("Number of columns:", df.shape[1])

    # Save raw data
    raw_file = project_path + "/raw_data/superstore_raw.csv"
    df.to_csv(raw_file, index=False)

    print("Raw data saved successfully!")

    return df


df = extract()

display(df.head())

Starting data extraction...
Data extracted successfully!
Number of records: 9994
Number of columns: 21
Raw data saved successfully!


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [4]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate records:", df.duplicated().sum())

display(df.describe())

Dataset shape: (9994, 21)

Column names:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,55190.379428,229.858001,3.789574,0.156203,28.656896
std,2885.163629,32063.693350,623.245101,2.225110,0.206452,234.260108
min,1.000000,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,56430.500000,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,90008.000000,209.940000,5.000000,0.200000,29.364000
max,9994.000000,99301.000000,22638.480000,14.000000,0.800000,8399.976000


In [5]:
# TRANSFORM

def transform(df):

    print("Starting data transformation...")

    # Create a copy of the original data
    df = df.copy()

    # 1. Rename column names
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

    print("1. Column names standardized")

    # 2. Remove duplicate records
    df = df.drop_duplicates()

    print("2. Duplicate records removed")

    # 3. Convert date columns
    df["order_date"] = pd.to_datetime(
        df["order_date"], errors="coerce"
    )

    df["ship_date"] = pd.to_datetime(
        df["ship_date"], errors="coerce"
    )

    print("3. Date columns converted")

    # 4. Convert numeric columns
    numeric_columns = [
        "sales", "quantity", "discount", "profit"
    ]

    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    print("4. Numeric columns converted")

    # 5. Handle missing values
    text_columns = df.select_dtypes(include="object").columns

    for col in text_columns:
        df[col] = df[col].fillna("Unknown")

    if "postal_code" in df.columns:
        df["postal_code"] = df["postal_code"].fillna("Unknown")

    print("5. Missing values handled")

    # 6. Standardize text values
    text_columns = [
        "ship_mode", "segment", "region",
        "category", "sub_category"
    ]

    for col in text_columns:
        df[col] = df[col].astype(str).str.strip().str.title()

    print("6. Text values standardized")

    # 7. Remove invalid records
    df = df.dropna(
        subset=["order_date", "ship_date", "sales", "quantity"]
    )

    df = df[
        (df["sales"] >= 0) &
        (df["quantity"] > 0) &
        (df["discount"].between(0, 1))
    ]

    print("7. Invalid records removed")

    # 8. Create shipping duration
    df["shipping_days"] = (
        df["ship_date"] - df["order_date"]
    ).dt.days

    df = df[df["shipping_days"] >= 0]

    print("8. Shipping duration created")

    # 9. Create year column
    df["order_year"] = df["order_date"].dt.year

    print("9. Order year created")

    # 10. Create month column
    df["order_month"] = df["order_date"].dt.month

    df["month_name"] = df["order_date"].dt.month_name()

    print("10. Month columns created")

    # 11. Calculate profit margin
    df["profit_margin"] = (
        df["profit"] / df["sales"].replace(0, np.nan)
    ) * 100

    print("11. Profit margin calculated")

    # 12. Sort data by order date
    df = df.sort_values(by="order_date")

    print("12. Data sorted by order date")

    print("\nTransformation completed successfully!")
    print("Final records:", df.shape[0])
    print("Final columns:", df.shape[1])

    return df


cleaned_df = transform(df)

display(cleaned_df.head())

Starting data transformation...
1. Column names standardized
2. Duplicate records removed
3. Date columns converted
4. Numeric columns converted
5. Missing values handled
6. Text values standardized
7. Invalid records removed
8. Shipping duration created
9. Order year created
10. Month columns created
11. Profit margin calculated
12. Data sorted by order date

Transformation completed successfully!
Final records: 9994
Final columns: 26


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,product_name,sales,quantity,discount,profit,shipping_days,order_year,order_month,month_name,profit_margin
7980,7981,CA-2014-103800,2014-01-03,2014-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",16.448,2,0.2,5.5512,4,2014,1,January,33.75
739,740,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,Avery 508,11.784,3,0.2,4.2717,4,2014,1,January,36.25
740,741,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748,4,2014,1,January,-23.75
741,742,CA-2014-112326,2014-01-04,2014-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870,4,2014,1,January,-155.00
1759,1760,CA-2014-141817,2014-01-05,2014-01-12,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,Avery Hi-Liter EverBold Pen Style Fluorescent ...,19.536,3,0.2,4.8840,7,2014,1,January,25.00


In [6]:
# LOAD

def load(df):

    print("Starting report generation...")

    # Report 1: Cleaned dataset
    df.to_csv(
        project_path + "/reports/cleaned_data.csv",
        index=False
    )

    print("1. Cleaned dataset saved")

    # Report 2: Category-wise sales
    category_sales = df.groupby("category").agg(
        total_sales=("sales", "sum"),
        total_profit=("profit", "sum"),
        total_quantity=("quantity", "sum")
    ).reset_index()

    category_sales.to_csv(
        project_path + "/reports/category_sales.csv",
        index=False
    )

    print("2. Category sales report saved")

    # Report 3: Region-wise sales
    region_sales = df.groupby("region").agg(
        total_sales=("sales", "sum"),
        total_profit=("profit", "sum")
    ).reset_index()

    region_sales.to_csv(
        project_path + "/reports/region_sales.csv",
        index=False
    )

    print("3. Region sales report saved")

    # Report 4: Monthly sales
    monthly_sales = df.groupby(
        ["order_year", "order_month", "month_name"]
    ).agg(
        total_sales=("sales", "sum"),
        total_profit=("profit", "sum")
    ).reset_index()

    monthly_sales.to_csv(
        project_path + "/reports/monthly_sales.csv",
        index=False
    )

    print("4. Monthly sales report saved")

    # Report 5: Top 10 products
    top_products = df.groupby("product_name").agg(
        total_sales=("sales", "sum"),
        total_profit=("profit", "sum")
    ).reset_index()

    top_products = top_products.sort_values(
        by="total_sales", ascending=False
    ).head(10)

    top_products.to_csv(
        project_path + "/reports/top10_products.csv",
        index=False
    )

    print("5. Top 10 products report saved")

    print("\nAll reports generated successfully!")

In [7]:
# RUN ETL PIPELINE

def run_etl_pipeline():

    print("================================")
    print("ETL PIPELINE STARTED")
    print("================================")

    # Extract
    raw_df = extract()

    # Transform
    transformed_df = transform(raw_df)

    # Load
    load(transformed_df)

    print("\n================================")
    print("ETL PIPELINE COMPLETED")
    print("================================")

    return transformed_df


final_df = run_etl_pipeline()

ETL PIPELINE STARTED
Starting data extraction...
Data extracted successfully!
Number of records: 9994
Number of columns: 21
Raw data saved successfully!
Starting data transformation...
1. Column names standardized
2. Duplicate records removed
3. Date columns converted
4. Numeric columns converted
5. Missing values handled
6. Text values standardized
7. Invalid records removed
8. Shipping duration created
9. Order year created
10. Month columns created
11. Profit margin calculated
12. Data sorted by order date

Transformation completed successfully!
Final records: 9994
Final columns: 26
Starting report generation...
1. Cleaned dataset saved
2. Category sales report saved
3. Region sales report saved
4. Monthly sales report saved
5. Top 10 products report saved

All reports generated successfully!

ETL PIPELINE COMPLETED


In [8]:
if __name__ == "__main__":
    run_etl_pipeline()

ETL PIPELINE STARTED
Starting data extraction...
Data extracted successfully!
Number of records: 9994
Number of columns: 21
Raw data saved successfully!
Starting data transformation...
1. Column names standardized
2. Duplicate records removed
3. Date columns converted
4. Numeric columns converted
5. Missing values handled
6. Text values standardized
7. Invalid records removed
8. Shipping duration created
9. Order year created
10. Month columns created
11. Profit margin calculated
12. Data sorted by order date

Transformation completed successfully!
Final records: 9994
Final columns: 26
Starting report generation...
1. Cleaned dataset saved
2. Category sales report saved
3. Region sales report saved
4. Monthly sales report saved
5. Top 10 products report saved

All reports generated successfully!

ETL PIPELINE COMPLETED
